In [1]:
from pathlib import Path
import json

PROJECT_ROOT = Path(
    "/home/sagemaker-user/AAI-540-Project/"
    "multilingual-it-ticket-routing-mlops"
)

config_path = PROJECT_ROOT / "project_config.json"
assert config_path.is_file(), "project_config.json was not found."

project_config = json.loads(config_path.read_text())

AWS_REGION = project_config["aws_region"]
PROJECT_BUCKET = project_config["s3_bucket"]

deployment = project_config["deployment"]
quality_report = project_config["monitoring"]["production_quality"]

assert deployment["status"] == "Completed", (
    "The saved batch deployment is not marked Completed."
)

print("Region:", AWS_REGION)
print("Project bucket:", PROJECT_BUCKET)
print("Completed batch job:", deployment["job_name"])
print("Production quality report:", quality_report["uri"])
print("\nMonitoring setup ready.")

Region: us-east-1
Project bucket: aai540-group3-tickets-390568313988-us-east-1
Completed batch job: aai540-group3-batch-20261004-045734
Production quality report: s3://aai540-group3-tickets-390568313988-us-east-1/monitoring/aai540-group3-batch-20261004-045734/production_quality.json

Monitoring setup ready.


In [2]:
from io import BytesIO
from urllib.parse import urlparse

import boto3
import pandas as pd

aws_session = boto3.Session(region_name=AWS_REGION)
s3_client = aws_session.client("s3")


def read_versioned_csv(file_info):
    location = urlparse(file_info["uri"])
    version_id = file_info["version_id"]

    assert version_id and version_id != "null", "Missing S3 version ID."

    response = s3_client.get_object(
        Bucket=location.netloc,
        Key=location.path.lstrip("/"),
        VersionId=version_id,
    )

    assert response["VersionId"] == version_id

    with response["Body"] as body:
        return pd.read_csv(
            BytesIO(body.read()),
            dtype={"ticket_id": "string"},
            low_memory=False,
        )


training_df = read_versioned_csv(project_config["s3_data"]["train"])
production_df = read_versioned_csv(
    project_config["s3_data"]["production"]
)

required_inputs = {"ticket_id", "subject", "body", "language"}

for name, frame in [
    ("Training", training_df),
    ("Production", production_df),
]:
    assert required_inputs.issubset(frame.columns)
    assert frame["ticket_id"].notna().all()
    assert frame["ticket_id"].is_unique
    print(f"{name}: {len(frame):,} tickets loaded")

assert len(training_df) == 17710
assert len(production_df) == 17712
assert "queue" not in production_df.columns
assert set(training_df["ticket_id"]).isdisjoint(
    set(production_df["ticket_id"])
)

print("\nVerified: separate datasets and no production queue labels.")
print("Ready to build the training-data reference.")

Training: 17,710 tickets loaded
Production: 17,712 tickets loaded

Verified: separate datasets and no production queue labels.
Ready to build the training-data reference.


In [3]:
import numpy as np


def monitoring_features(frame):
    subject = frame["subject"].fillna("").astype(str).str.strip()
    body = frame["body"].fillna("").astype(str).str.strip()
    text = (subject + " " + body).str.strip()

    language = (
        frame["language"]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .str.lower()
        .replace("", "unknown")
    )

    return pd.DataFrame({
        "language": language,
        "text_char_count": text.str.len(),
        "text_word_count": text.str.split().str.len(),
        "missing_subject": subject.eq(""),
        "missing_body": body.eq(""),
        "empty_text": text.eq(""),
        "unknown_language": language.eq("unknown"),
    }, index=frame.index)


training_monitor_features = monitoring_features(training_df)

training_baseline = {
    "reference_split": "train",
    "source": project_config["s3_data"]["train"],
    "rows": len(training_df),
    "language_proportions": (
        training_monitor_features["language"]
        .value_counts(normalize=True)
        .sort_index()
        .to_dict()
    ),
    "missing_rates": {},
    "numeric_features": {},
}

for column in [
    "missing_subject", "missing_body", "empty_text", "unknown_language"
]:
    training_baseline["missing_rates"][column] = float(
        training_monitor_features[column].mean()
    )

for column in ["text_char_count", "text_word_count"]:
    values = training_monitor_features[column].to_numpy()

    # Define bins using training data only.
    # The first and last bins also cover future values outside its range.
    cut_points = np.unique(
        np.quantile(values, np.arange(0.1, 1.0, 0.1))
    )
    bin_indices = np.searchsorted(cut_points, values, side="right")
    bin_counts = np.bincount(
        bin_indices, minlength=len(cut_points) + 1
    )

    training_baseline["numeric_features"][column] = {
        "mean": float(values.mean()),
        "median": float(np.median(values)),
        "p95": float(np.quantile(values, 0.95)),
        "cut_points": cut_points.tolist(),
        "bin_proportions": (bin_counts / len(values)).tolist(),
    }

monitoring_report_dir = PROJECT_ROOT / "reports" / "monitoring"
monitoring_report_dir.mkdir(parents=True, exist_ok=True)

baseline_path = monitoring_report_dir / "training_data_baseline.json"
baseline_path.write_text(
    json.dumps(training_baseline, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Reference tickets:", training_baseline["rows"])
print("Languages:", sorted(training_baseline["language_proportions"]))
print("Missing-text rates:", training_baseline["missing_rates"])
print("Training reference saved:", baseline_path)

Reference tickets: 17710
Languages: ['de', 'en', 'es', 'fr', 'pt']
Missing-text rates: {'missing_subject': 0.10316205533596838, 'missing_body': 0.0, 'empty_text': 0.0, 'unknown_language': 0.0}
Training reference saved: /home/sagemaker-user/AAI-540-Project/multilingual-it-ticket-routing-mlops/reports/monitoring/training_data_baseline.json


In [4]:
# Set thresholds before calculating production monitoring results.
data_thresholds = {
    "distribution_distance": 0.10,
    "missing_rate_increase": 0.05,
    "empty_text_rate": 0.0,
    "unknown_language_rate": 0.0,
    "unseen_language_rate": 0.0,
}

production_monitor_features = monitoring_features(production_df)
data_checks = []


def add_check(name, value, threshold):
    data_checks.append({
        "check": name,
        "value": float(value),
        "threshold": float(threshold),
        "status": "ALERT" if value > threshold else "OK",
    })


# Total variation distance: 0 means identical distributions;
# 1 means completely non-overlapping distributions.
reference_languages = training_baseline["language_proportions"]
current_languages = (
    production_monitor_features["language"]
    .value_counts(normalize=True)
    .to_dict()
)

all_languages = sorted(
    set(reference_languages) | set(current_languages)
)

language_distance = 0.5 * sum(
    abs(
        reference_languages.get(language, 0.0)
        - current_languages.get(language, 0.0)
    )
    for language in all_languages
)

add_check(
    "language_distribution",
    language_distance,
    data_thresholds["distribution_distance"],
)

# Reuse the training bins; do not create new bins from production.
for column, reference in training_baseline["numeric_features"].items():
    values = production_monitor_features[column].to_numpy()
    bin_indices = np.searchsorted(
        reference["cut_points"], values, side="right"
    )
    counts = np.bincount(
        bin_indices,
        minlength=len(reference["bin_proportions"]),
    )
    proportions = counts / len(values)

    distance = 0.5 * np.abs(
        proportions - np.array(reference["bin_proportions"])
    ).sum()

    add_check(
        f"{column}_distribution",
        distance,
        data_thresholds["distribution_distance"],
    )

for column in ["missing_subject", "missing_body"]:
    increase = max(
        0.0,
        float(production_monitor_features[column].mean())
        - training_baseline["missing_rates"][column],
    )
    add_check(
        f"{column}_rate_increase",
        increase,
        data_thresholds["missing_rate_increase"],
    )

for column in ["empty_text", "unknown_language"]:
    add_check(
        f"{column}_rate",
        production_monitor_features[column].mean(),
        data_thresholds[f"{column}_rate"],
    )

unseen_language_rate = (
    ~production_monitor_features["language"].isin(reference_languages)
).mean()

add_check(
    "unseen_language_rate",
    unseen_language_rate,
    data_thresholds["unseen_language_rate"],
)

data_quality_report = {
    "reference_source": training_baseline["source"],
    "production_source": project_config["s3_data"]["production"],
    "production_rows": len(production_df),
    "thresholds": data_thresholds,
    "checks": data_checks,
    "alert_count": sum(item["status"] == "ALERT" for item in data_checks),
}

(monitoring_report_dir / "production_data_quality.json").write_text(
    json.dumps(data_quality_report, indent=2, allow_nan=False),
    encoding="utf-8",
)

display(pd.DataFrame(data_checks).round(4))
print("Alerts:", data_quality_report["alert_count"])
print("Production data monitoring report saved.")

,check,value,threshold,status
0,language_distribution,0.0042,0.10,OK
1,text_char_count_distribution,0.0127,0.10,OK
2,text_word_count_distribution,0.0103,0.10,OK
3,missing_subject_rate_increase,0.0008,0.05,OK
4,missing_body_rate_increase,0.0001,0.05,OK
5,empty_text_rate,0.0000,0.00,OK
6,unknown_language_rate,0.0000,0.00,OK
7,unseen_language_rate,0.0000,0.00,OK


Alerts: 0
Production data monitoring report saved.


In [5]:
# Load the saved production quality report from its exact S3 version.
report_location = urlparse(quality_report["uri"])

response = s3_client.get_object(
    Bucket=report_location.netloc,
    Key=report_location.path.lstrip("/"),
    VersionId=quality_report["version_id"],
)

assert response["VersionId"] == quality_report["version_id"]

with response["Body"] as body:
    production_quality = json.loads(body.read())

assert production_quality["batch_job_name"] == deployment["job_name"]

current_metrics = production_quality["metrics"]
reference_metrics = project_config["final_model"]["test_metrics"]

assert current_metrics["tickets"] == len(production_df)

# Initial demonstration threshold: an absolute drop of 0.05.
allowed_drop = 0.05
model_checks = []

for metric in ["accuracy", "macro_f1", "weighted_f1"]:
    reference_value = float(reference_metrics[metric])
    current_value = float(current_metrics[metric])

    assert 0.0 <= reference_value <= 1.0
    assert 0.0 <= current_value <= 1.0

    minimum_value = max(0.0, reference_value - allowed_drop)

    model_checks.append({
        "metric": metric,
        "test_reference": reference_value,
        "production_value": current_value,
        "minimum_allowed": minimum_value,
        "status": "ALERT" if current_value < minimum_value else "OK",
    })

model_quality_report = {
    "batch_job_name": deployment["job_name"],
    "training_job_name": project_config["final_model"]["training_job_name"],
    "reference_split": "test",
    "reference_source": project_config["final_model"]["test_dataset"],
    "production_report_source": {
        "uri": quality_report["uri"],
        "version_id": quality_report["version_id"],
    },
    "allowed_absolute_drop": allowed_drop,
    "checks": model_checks,
    "alert_count": sum(
        item["status"] == "ALERT" for item in model_checks
    ),
}

(monitoring_report_dir / "model_quality_checks.json").write_text(
    json.dumps(model_quality_report, indent=2, allow_nan=False),
    encoding="utf-8",
)

display(pd.DataFrame(model_checks).round(4))
print("Model quality alerts:", model_quality_report["alert_count"])
print("Model quality monitoring report saved.")

,metric,test_reference,production_value,minimum_allowed,status
0,accuracy,0.4634,0.4591,0.4134,OK
1,macro_f1,0.4464,0.4380,0.3964,OK
2,weighted_f1,0.4653,0.4605,0.4153,OK


Model quality alerts: 0
Model quality monitoring report saved.


In [6]:
from datetime import datetime, timezone

sagemaker_client = aws_session.client("sagemaker")

training_details = sagemaker_client.describe_training_job(
    TrainingJobName=project_config["final_model"]["training_job_name"]
)

batch_details = sagemaker_client.describe_transform_job(
    TransformJobName=deployment["job_name"]
)


def summarize_job(details, job_type):
    if job_type == "training":
        name = details["TrainingJobName"]
        status = details["TrainingJobStatus"]
        start = details.get("TrainingStartTime")
        end = details.get("TrainingEndTime")
    else:
        name = details["TransformJobName"]
        status = details["TransformJobStatus"]
        start = details.get("TransformStartTime")
        end = details.get("TransformEndTime")

    duration = (
        (end - start).total_seconds()
        if start is not None and end is not None
        else None
    )

    if status == "Completed":
        check_status = "OK"
    elif status in {"Failed", "Stopped", "Stopping"}:
        check_status = "ALERT"
    else:
        check_status = "PENDING"

    return {
        "job_type": job_type,
        "job_name": name,
        "aws_status": status,
        "check_status": check_status,
        "duration_seconds": duration,
        "start_time": start.isoformat() if start else None,
        "end_time": end.isoformat() if end else None,
        "failure_reason": details.get("FailureReason"),
    }


infrastructure_jobs = [
    summarize_job(training_details, "training"),
    summarize_job(batch_details, "batch_prediction"),
]

infrastructure_report = {
    "checked_at": datetime.now(timezone.utc).isoformat(),
    "jobs": infrastructure_jobs,
    "alert_count": sum(
        job["check_status"] == "ALERT" for job in infrastructure_jobs
    ),
}

(monitoring_report_dir / "infrastructure_status.json").write_text(
    json.dumps(infrastructure_report, indent=2, allow_nan=False),
    encoding="utf-8",
)

display(pd.DataFrame(infrastructure_jobs)[[
    "job_type",
    "aws_status",
    "duration_seconds",
    "check_status",
]])

print("Infrastructure alerts:", infrastructure_report["alert_count"])
print("Infrastructure status report saved.")

,job_type,aws_status,duration_seconds,check_status
0,training,Completed,118.783,OK
1,batch_prediction,Completed,72.528,OK


Infrastructure alerts: 0
Infrastructure status report saved.


In [7]:
cloudwatch_client = aws_session.client("cloudwatch")

METRIC_NAMESPACE = "AAI540/TicketRouting"
metric_dimensions = [
    {"Name": "Project", "Value": "aai540-group3"},
    {"Name": "Environment", "Value": "simulated-production"},
]

snapshot_time = datetime.now(timezone.utc)
metric_data = []


def add_metric(name, value, unit="None"):
    metric_data.append({
        "MetricName": name,
        "Dimensions": metric_dimensions,
        "Timestamp": snapshot_time,
        "Value": float(value),
        "Unit": unit,
    })


# Model performance: values remain on a 0–1 scale.
for metric, name in [
    ("accuracy", "ProductionAccuracy"),
    ("macro_f1", "ProductionMacroF1"),
    ("weighted_f1", "ProductionWeightedF1"),
]:
    add_metric(name, current_metrics[metric])

# Number of checks that triggered alerts.
add_metric("DataAlertCount", data_quality_report["alert_count"], "Count")
add_metric(
    "ModelQualityAlertCount", model_quality_report["alert_count"], "Count"
)
add_metric(
    "InfrastructureAlertCount",
    infrastructure_report["alert_count"],
    "Count",
)

# Individual input-data checks.
data_metric_names = {
    "language_distribution": "LanguageDistributionDistance",
    "text_char_count_distribution": "TextCharDistributionDistance",
    "text_word_count_distribution": "TextWordDistributionDistance",
    "missing_subject_rate_increase": "MissingSubjectRateIncrease",
    "missing_body_rate_increase": "MissingBodyRateIncrease",
    "empty_text_rate": "EmptyTextRate",
    "unknown_language_rate": "UnknownLanguageRate",
    "unseen_language_rate": "UnseenLanguageRate",
}

for check in data_checks:
    add_metric(data_metric_names[check["check"]], check["value"])

# Job runtimes and status checks.
for job in infrastructure_jobs:
    prefix = "Training" if job["job_type"] == "training" else "Batch"

    if job["duration_seconds"] is not None:
        add_metric(
            f"{prefix}DurationSeconds",
            job["duration_seconds"],
            "Seconds",
        )

    add_metric(
        f"{prefix}JobAlert",
        int(job["check_status"] == "ALERT"),
        "Count",
    )

result = cloudwatch_client.put_metric_data(
    Namespace=METRIC_NAMESPACE,
    MetricData=metric_data,
)

assert result["ResponseMetadata"]["HTTPStatusCode"] == 200

project_config["monitoring"]["cloudwatch"] = {
    "namespace": METRIC_NAMESPACE,
    "dimensions": metric_dimensions,
    "last_published_at": snapshot_time.isoformat(),
    "batch_job_name": deployment["job_name"],
    "metric_names": [item["MetricName"] for item in metric_data],
}

config_path.write_text(
    json.dumps(project_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("CloudWatch accepted metrics:", len(metric_data))
print("Namespace:", METRIC_NAMESPACE)
print("Monitoring configuration saved.")

CloudWatch accepted metrics: 18
Namespace: AAI540/TicketRouting
Monitoring configuration saved.


In [8]:
dashboard_name = "AAI540-Group3-Ticket-Monitoring"

# Each dashboard metric must use the same dimensions as publication.
dimension_fields = [
    value
    for dimension in metric_dimensions
    for value in (dimension["Name"], dimension["Value"])
]


def metric_widget(title, metric_names, x, y, width=12):
    return {
        "type": "metric",
        "x": x,
        "y": y,
        "width": width,
        "height": 6,
        "properties": {
            "title": title,
            "region": AWS_REGION,
            "view": "singleValue",
            "sparkline": False,
            "stat": "Average",
            "period": 300,
            "metrics": [
                [METRIC_NAMESPACE, name, *dimension_fields]
                for name in metric_names
            ],
        },
    }


dashboard_body = {
    "start": "-P7D",
    "periodOverride": "inherit",
    "widgets": [
        {
            "type": "text",
            "x": 0,
            "y": 0,
            "width": 24,
            "height": 4,
            "properties": {
                "markdown": (
                    "# Multilingual ticket routing — monitoring\n"
                    f"Batch job: `{deployment['job_name']}`\n\n"
                    f"Snapshot published: `{snapshot_time.isoformat()}`\n\n"
                    "Metrics are published when monitoring runs. "
                    "This is not yet scheduled monitoring. "
                    "Quality scores and data rates use a 0–1 scale. "
                    "Zero alerts means the configured checks passed; "
                    "it does not establish readiness for automatic routing."
                )
            },
        },
        metric_widget(
            "Production model quality — higher is better",
            [
                "ProductionAccuracy",
                "ProductionMacroF1",
                "ProductionWeightedF1",
            ],
            0, 4,
        ),
        metric_widget(
            "Triggered checks — zero means no alerts",
            [
                "DataAlertCount",
                "ModelQualityAlertCount",
                "InfrastructureAlertCount",
            ],
            12, 4,
        ),
        metric_widget(
            "Data distribution changes — alert above 0.10",
            [
                "LanguageDistributionDistance",
                "TextCharDistributionDistance",
                "TextWordDistributionDistance",
            ],
            0, 10,
        ),
        metric_widget(
            "AWS job duration — seconds",
            ["TrainingDurationSeconds", "BatchDurationSeconds"],
            12, 10,
        ),
        metric_widget(
            "Input quality — missing-rate increases and invalid-input rates",
            [
                "MissingSubjectRateIncrease",
                "MissingBodyRateIncrease",
                "EmptyTextRate",
                "UnknownLanguageRate",
                "UnseenLanguageRate",
            ],
            0, 16, width=24,
        ),
    ],
}

dashboard_json = json.dumps(dashboard_body, indent=2)

(monitoring_report_dir / "cloudwatch_dashboard.json").write_text(
    dashboard_json, encoding="utf-8"
)

result = cloudwatch_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=dashboard_json,
)

validation_messages = result.get("DashboardValidationMessages", [])
if validation_messages:
    raise RuntimeError(
        "Dashboard validation messages: "
        + json.dumps(validation_messages, indent=2)
    )

dashboard_url = (
    f"https://console.aws.amazon.com/cloudwatch/home?region={AWS_REGION}"
    f"#dashboards:name={dashboard_name}"
)

project_config["monitoring"]["cloudwatch"].update({
    "dashboard_name": dashboard_name,
    "dashboard_url": dashboard_url,
})

config_path.write_text(
    json.dumps(project_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Dashboard created:", dashboard_name)
print("Open this link while signed into your AWS lab:")
print(dashboard_url)

Dashboard created: AAI540-Group3-Ticket-Monitoring
Open this link while signed into your AWS lab:
https://console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=AAI540-Group3-Ticket-Monitoring


In [9]:
from datetime import timedelta

published_at = datetime.fromisoformat(
    project_config["monitoring"]["cloudwatch"]["last_published_at"]
)

metric_check = cloudwatch_client.get_metric_statistics(
    Namespace=METRIC_NAMESPACE,
    MetricName="ProductionAccuracy",
    Dimensions=metric_dimensions,
    StartTime=published_at - timedelta(minutes=5),
    EndTime=published_at + timedelta(minutes=5),
    Period=60,
    Statistics=["Average"],
)

points = sorted(
    metric_check["Datapoints"],
    key=lambda point: point["Timestamp"],
)

print("Accuracy data points found:", len(points))

for point in points:
    print(
        "Timestamp:", point["Timestamp"].isoformat(),
        "| Accuracy:", round(point["Average"], 4),
    )

Accuracy data points found: 1
Timestamp: 2026-10-04T05:32:00+00:00 | Accuracy: 0.4591


In [10]:
# Pin the dashboard to the saved monitoring snapshot.
window_start = published_at - timedelta(minutes=5)
window_end = published_at + timedelta(minutes=5)

start_text = window_start.strftime("%Y-%m-%dT%H:%M:%SZ")
end_text = window_end.strftime("%Y-%m-%dT%H:%M:%SZ")

dashboard_body["start"] = start_text
dashboard_body["end"] = end_text

for widget in dashboard_body["widgets"]:
    if widget["type"] == "metric":
        widget["properties"].update({
            "start": start_text,
            "end": end_text,
            "setPeriodToTimeRange": True,
            "sparkline": False,
        })

dashboard_json = json.dumps(dashboard_body, indent=2)

result = cloudwatch_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=dashboard_json,
)

messages = result.get("DashboardValidationMessages", [])
if messages:
    raise RuntimeError(json.dumps(messages, indent=2))

(monitoring_report_dir / "cloudwatch_dashboard.json").write_text(
    dashboard_json, encoding="utf-8"
)

project_config["monitoring"]["cloudwatch"].update({
    "dashboard_window_start": start_text,
    "dashboard_window_end": end_text,
})

config_path.write_text(
    json.dumps(project_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Dashboard updated to show the saved batch snapshot.")
print(dashboard_url)

Dashboard updated to show the saved batch snapshot.
https://console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=AAI540-Group3-Ticket-Monitoring


In [11]:
alarm_definitions = {
    "AAI540-Group3-DataQuality": "DataAlertCount",
    "AAI540-Group3-ModelQuality": "ModelQualityAlertCount",
    "AAI540-Group3-Infrastructure": "InfrastructureAlertCount",
}

for alarm_name, metric_name in alarm_definitions.items():
    cloudwatch_client.put_metric_alarm(
        AlarmName=alarm_name,
        AlarmDescription=(
            f"Flag when {metric_name} exceeds zero. "
            "Metrics are published by the ticket monitoring workflow."
        ),
        Namespace=METRIC_NAMESPACE,
        MetricName=metric_name,
        Dimensions=metric_dimensions,
        Statistic="Maximum",
        Period=3600,
        EvaluationPeriods=1,
        DatapointsToAlarm=1,
        Threshold=0.0,
        ComparisonOperator="GreaterThanThreshold",
        TreatMissingData="missing",
        ActionsEnabled=False,
    )
    print("Alarm created:", alarm_name)

alarm_response = cloudwatch_client.describe_alarms(
    AlarmNames=list(alarm_definitions)
)

alarm_records = [
    {
        "name": alarm["AlarmName"],
        "arn": alarm["AlarmArn"],
        "metric": alarm["MetricName"],
        "state": alarm["StateValue"],
    }
    for alarm in alarm_response["MetricAlarms"]
]

assert len(alarm_records) == 3

project_config["monitoring"]["cloudwatch"]["alarms"] = alarm_records

config_path.write_text(
    json.dumps(project_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

display(pd.DataFrame(alarm_records)[["name", "metric", "state"]])
print("Alarm configuration saved.")

Alarm created: AAI540-Group3-DataQuality
Alarm created: AAI540-Group3-ModelQuality
Alarm created: AAI540-Group3-Infrastructure


,name,metric,state
0,AAI540-Group3-DataQuality,DataAlertCount,INSUFFICIENT_DATA
1,AAI540-Group3-Infrastructure,InfrastructureAlertCount,INSUFFICIENT_DATA
2,AAI540-Group3-ModelQuality,ModelQualityAlertCount,INSUFFICIENT_DATA


Alarm configuration saved.


In [12]:
import hashlib

# Refresh the alarm states without recreating the alarms.
alarm_response = cloudwatch_client.describe_alarms(
    AlarmNames=list(alarm_definitions)
)

alarm_records = [
    {
        "name": alarm["AlarmName"],
        "arn": alarm["AlarmArn"],
        "metric": alarm["MetricName"],
        "state": alarm["StateValue"],
    }
    for alarm in alarm_response["MetricAlarms"]
]

alarm_report = {
    "checked_at": datetime.now(timezone.utc).isoformat(),
    "alarms": alarm_records,
}

(monitoring_report_dir / "alarm_status.json").write_text(
    json.dumps(alarm_report, indent=2),
    encoding="utf-8",
)

report_names = [
    "training_data_baseline.json",
    "production_data_quality.json",
    "model_quality_checks.json",
    "infrastructure_status.json",
    "cloudwatch_dashboard.json",
    "alarm_status.json",
]

saved_artifacts = {}

for filename in report_names:
    report_path = monitoring_report_dir / filename
    report_bytes = report_path.read_bytes()
    object_key = f"monitoring/{deployment['job_name']}/{filename}"

    upload = s3_client.put_object(
        Bucket=PROJECT_BUCKET,
        Key=object_key,
        Body=report_bytes,
        ContentType="application/json",
    )

    version_id = upload.get("VersionId")
    assert version_id and version_id != "null"

    saved_artifacts[report_path.stem] = {
        "uri": f"s3://{PROJECT_BUCKET}/{object_key}",
        "version_id": version_id,
        "sha256": hashlib.sha256(report_bytes).hexdigest(),
    }

    print("Saved to S3:", filename)

monitoring_config = project_config["monitoring"]
monitoring_config.setdefault("artifacts", {}).update(saved_artifacts)
monitoring_config["data_thresholds"] = data_thresholds
monitoring_config["model_allowed_absolute_drop"] = allowed_drop
monitoring_config["cloudwatch"]["alarms"] = alarm_records

config_path.write_text(
    json.dumps(project_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

display(pd.DataFrame(alarm_records)[["name", "state"]])
print("Versioned monitoring reports and configuration saved.")

Saved to S3: training_data_baseline.json
Saved to S3: production_data_quality.json
Saved to S3: model_quality_checks.json
Saved to S3: infrastructure_status.json
Saved to S3: cloudwatch_dashboard.json
Saved to S3: alarm_status.json


,name,state
0,AAI540-Group3-DataQuality,INSUFFICIENT_DATA
1,AAI540-Group3-Infrastructure,OK
2,AAI540-Group3-ModelQuality,INSUFFICIENT_DATA


Versioned monitoring reports and configuration saved.


In [13]:
MONITORING_SOURCE_DIR = PROJECT_ROOT / "src" / "ticket_monitoring"
MONITORING_SOURCE_DIR.mkdir(parents=True, exist_ok=True)

monitoring_job_config = {
    "aws_region": AWS_REGION,
    "training_job_name": project_config["final_model"]["training_job_name"],
    "batch_job_name": deployment["job_name"],

    # Exact saved versions of the reference, inputs, predictions, and labels.
    "training_baseline": project_config["monitoring"]["artifacts"][
        "training_data_baseline"
    ],
    "production_data": project_config["s3_data"]["production"],
    "production_labels": project_config["s3_data"]["production_labels"],
    "prediction_artifacts": deployment["prediction_artifacts"],

    "expected_production_rows": len(production_df),
    "queue_labels": sorted(training_df["queue"].unique().tolist()),

    # Keep the same monitoring rules used in the notebook.
    "reference_metrics": project_config["final_model"]["test_metrics"],
    "data_thresholds": data_thresholds,
    "model_allowed_absolute_drop": allowed_drop,

    "cloudwatch": {
        "namespace": METRIC_NAMESPACE,
        "dimensions": metric_dimensions,
    },
}

monitoring_config_path = (
    MONITORING_SOURCE_DIR / "monitoring_config.json"
)

monitoring_config_path.write_text(
    json.dumps(monitoring_job_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Monitoring script folder:", MONITORING_SOURCE_DIR)
print("Job configuration saved:", monitoring_config_path.name)
print("Expected production tickets:",
      monitoring_job_config["expected_production_rows"])
print("Queue labels:", len(monitoring_job_config["queue_labels"]))

Monitoring script folder: /home/sagemaker-user/AAI-540-Project/multilingual-it-ticket-routing-mlops/src/ticket_monitoring
Job configuration saved: monitoring_config.json
Expected production tickets: 17712
Queue labels: 10


In [14]:
%%writefile src/ticket_monitoring/checks.py
"""Reusable data and model quality checks for ticket monitoring."""

from bisect import bisect_right
from collections import Counter
import math


def data_quality_checks(records, baseline, thresholds):
    if not records:
        raise ValueError("Production data is empty.")

    required = {"ticket_id", "subject", "body", "language"}
    languages = Counter()
    char_counts = []
    word_counts = []
    missing = Counter()

    for record in records:
        if not required.issubset(record):
            raise ValueError("A production record is missing required fields.")

        subject = (record["subject"] or "").strip()
        body = (record["body"] or "").strip()
        text = (subject + " " + body).strip()
        language = (record["language"] or "").strip().lower() or "unknown"

        languages[language] += 1
        char_counts.append(len(text))
        word_counts.append(len(text.split()))
        missing["missing_subject"] += int(not subject)
        missing["missing_body"] += int(not body)
        missing["empty_text"] += int(not text)
        missing["unknown_language"] += int(language == "unknown")

    count = len(records)
    checks = []

    def add(name, value, threshold):
        checks.append({
            "check": name,
            "value": float(value),
            "threshold": float(threshold),
            "status": "ALERT" if value > threshold else "OK",
        })

    reference_languages = baseline["language_proportions"]
    all_languages = set(reference_languages) | set(languages)

    distance = 0.5 * sum(
        abs(reference_languages.get(language, 0.0)
            - languages.get(language, 0) / count)
        for language in all_languages
    )
    add("language_distribution", distance,
        thresholds["distribution_distance"])

    for column, values in [
        ("text_char_count", char_counts),
        ("text_word_count", word_counts),
    ]:
        reference = baseline["numeric_features"][column]
        bin_counts = [0] * len(reference["bin_proportions"])

        for value in values:
            index = bisect_right(reference["cut_points"], value)
            bin_counts[index] += 1

        distance = 0.5 * sum(
            abs(observed / count - expected)
            for observed, expected in zip(
                bin_counts, reference["bin_proportions"]
            )
        )
        add(f"{column}_distribution", distance,
            thresholds["distribution_distance"])

    for column in ["missing_subject", "missing_body"]:
        increase = max(
            0.0,
            missing[column] / count - baseline["missing_rates"][column],
        )
        add(f"{column}_rate_increase", increase,
            thresholds["missing_rate_increase"])

    for column in ["empty_text", "unknown_language"]:
        add(f"{column}_rate", missing[column] / count,
            thresholds[f"{column}_rate"])

    unseen_count = sum(
        number for language, number in languages.items()
        if language not in reference_languages
    )
    add("unseen_language_rate", unseen_count / count,
        thresholds["unseen_language_rate"])

    return checks


def classification_metrics(actual, predicted, queue_labels):
    """Calculate accuracy and F1 across the fixed set of routing queues."""
    if not actual or len(actual) != len(predicted):
        raise ValueError("Labels and predictions must have equal, nonzero length.")

    if not queue_labels or len(set(queue_labels)) != len(queue_labels):
        raise ValueError("Queue labels must be nonempty and unique.")

    allowed = set(queue_labels)
    if not set(actual).issubset(allowed):
        raise ValueError("Ground truth contains an unexpected queue.")
    if not set(predicted).issubset(allowed):
        raise ValueError("Predictions contain an unexpected queue.")

    actual_counts = Counter(actual)
    predicted_counts = Counter(predicted)
    correct_counts = Counter(
        truth for truth, prediction in zip(actual, predicted)
        if truth == prediction
    )

    f1_scores = {}
    for queue in queue_labels:
        denominator = actual_counts[queue] + predicted_counts[queue]
        f1_scores[queue] = (
            2.0 * correct_counts[queue] / denominator
            if denominator else 0.0
        )

    count = len(actual)
    return {
        "accuracy": sum(correct_counts.values()) / count,
        "macro_f1": sum(f1_scores.values()) / len(queue_labels),
        "weighted_f1": sum(
            f1_scores[queue] * actual_counts[queue]
            for queue in queue_labels
        ) / count,
    }


def model_quality_checks(metrics, reference_metrics, allowed_drop):
    if not 0.0 <= allowed_drop <= 1.0:
        raise ValueError("Allowed drop must be between zero and one.")

    checks = []
    for metric in ["accuracy", "macro_f1", "weighted_f1"]:
        current = float(metrics[metric])
        reference = float(reference_metrics[metric])

        if not all(
            math.isfinite(value) and 0.0 <= value <= 1.0
            for value in (current, reference)
        ):
            raise ValueError(f"Invalid value for {metric}.")

        minimum = max(0.0, reference - allowed_drop)
        checks.append({
            "metric": metric,
            "test_reference": reference,
            "production_value": current,
            "minimum_allowed": minimum,
            "status": "ALERT" if current < minimum else "OK",
        })

    return checks

Writing src/ticket_monitoring/checks.py


In [1]:
from pathlib import Path
import os
import json
import hashlib
import importlib.util

import boto3

PROJECT_ROOT = Path(
    "/home/sagemaker-user/AAI-540-Project/"
    "multilingual-it-ticket-routing-mlops"
)
assert PROJECT_ROOT.is_dir(), "Project folder was not found."
os.chdir(PROJECT_ROOT)

config_path = PROJECT_ROOT / "project_config.json"
project_config = json.loads(config_path.read_text(encoding="utf-8"))

AWS_REGION = project_config["aws_region"]
PROJECT_BUCKET = project_config["s3_bucket"]

MONITORING_SOURCE_DIR = PROJECT_ROOT / "src" / "ticket_monitoring"
monitoring_report_dir = PROJECT_ROOT / "reports" / "monitoring"

monitoring_job_config = json.loads(
    (MONITORING_SOURCE_DIR / "monitoring_config.json")
    .read_text(encoding="utf-8")
)

# Restore and verify the saved training reference.
baseline_bytes = (
    monitoring_report_dir / "training_data_baseline.json"
).read_bytes()

assert hashlib.sha256(baseline_bytes).hexdigest() == (
    monitoring_job_config["training_baseline"]["sha256"]
), "The local training reference differs from the saved version."

training_baseline = json.loads(baseline_bytes)
data_thresholds = monitoring_job_config["data_thresholds"]
allowed_drop = monitoring_job_config["model_allowed_absolute_drop"]
reference_metrics = monitoring_job_config["reference_metrics"]

# Import the reusable functions. This does not launch a job.
checks_path = MONITORING_SOURCE_DIR / "checks.py"
spec = importlib.util.spec_from_file_location(
    "ticket_monitoring_checks", checks_path
)
checks_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(checks_module)

aws_session = boto3.Session(region_name=AWS_REGION)
s3_client = aws_session.client("s3")

print("Project folder:", PROJECT_ROOT)
print("Region:", AWS_REGION)
print("Training reference fingerprint verified.")
print("Reusable monitoring checks imported.")
print(
    "Expected production tickets:",
    monitoring_job_config["expected_production_rows"],
)
print("Session restored. No AWS jobs started.")

Project folder: /home/sagemaker-user/AAI-540-Project/multilingual-it-ticket-routing-mlops
Region: us-east-1
Training reference fingerprint verified.
Reusable monitoring checks imported.
Expected production tickets: 17712
Session restored. No AWS jobs started.


In [2]:
import csv
import io
import math
from urllib.parse import urlparse


def read_saved_s3_bytes(file_info):
    location = urlparse(file_info["uri"])
    version_id = file_info["version_id"]
    assert version_id and version_id != "null"

    response = s3_client.get_object(
        Bucket=location.netloc,
        Key=location.path.lstrip("/"),
        VersionId=version_id,
    )
    assert response["VersionId"] == version_id

    with response["Body"] as body:
        content = body.read()

    if file_info.get("sha256"):
        assert hashlib.sha256(content).hexdigest() == file_info["sha256"]

    return content


production_bytes = read_saved_s3_bytes(
    monitoring_job_config["production_data"]
)

production_records = list(csv.DictReader(
    io.StringIO(production_bytes.decode("utf-8-sig"))
))

assert len(production_records) == (
    monitoring_job_config["expected_production_rows"]
)
assert all(record["ticket_id"] for record in production_records)
assert len({r["ticket_id"] for r in production_records}) == len(
    production_records
)
assert all("queue" not in record for record in production_records)

# Compare reusable checks with the original notebook results.
verified_data_checks = checks_module.data_quality_checks(
    production_records, training_baseline, data_thresholds
)

saved_data_report = json.loads(
    read_saved_s3_bytes(
        project_config["monitoring"]["artifacts"]["production_data_quality"]
    )
)
expected_checks = {
    item["check"]: item for item in saved_data_report["checks"]
}

assert {item["check"] for item in verified_data_checks} == set(
    expected_checks
)

for check in verified_data_checks:
    expected = expected_checks[check["check"]]
    assert math.isclose(
        check["value"], expected["value"],
        rel_tol=1e-9, abs_tol=1e-12,
    ), f"Value mismatch: {check['check']}"
    assert check["threshold"] == expected["threshold"]
    assert check["status"] == expected["status"]

print("PASS: all 8 data checks match the saved notebook results.")

# Deliberately damage a COPY; the production data stays unchanged.
bad_records = [dict(record) for record in production_records[:100]]
for record in bad_records:
    record.update(subject="", body="", language="zz")

bad_checks = checks_module.data_quality_checks(
    bad_records, training_baseline, data_thresholds
)
bad_status = {item["check"]: item["status"] for item in bad_checks}

for name in [
    "empty_text_rate",
    "unseen_language_rate",
    "language_distribution",
]:
    assert bad_status[name] == "ALERT", f"Missed expected alert: {name}"

print("PASS: empty tickets triggered an alert.")
print("PASS: an unfamiliar language triggered an alert.")
print("PASS: the changed language distribution triggered an alert.")
print("No test data or test metrics were uploaded to AWS.")

PASS: all 8 data checks match the saved notebook results.
PASS: empty tickets triggered an alert.
PASS: an unfamiliar language triggered an alert.
PASS: the changed language distribution triggered an alert.
No test data or test metrics were uploaded to AWS.


In [3]:
from datetime import datetime, timezone

# Retrieve the exact saved prediction files.
prediction_records = []
for artifact in monitoring_job_config["prediction_artifacts"]:
    content = read_saved_s3_bytes(artifact).decode("utf-8")
    prediction_records.extend(
        json.loads(line)
        for line in content.splitlines()
        if line.strip()
    )

label_bytes = read_saved_s3_bytes(
    monitoring_job_config["production_labels"]
)
label_records = list(csv.DictReader(
    io.StringIO(label_bytes.decode("utf-8-sig"))
))


def index_by_ticket(records):
    indexed = {}
    for record in records:
        ticket_id = record["ticket_id"]
        assert ticket_id, "Empty ticket ID."
        assert ticket_id not in indexed, "Duplicate ticket ID."
        indexed[ticket_id] = record
    return indexed


predictions_by_id = index_by_ticket(prediction_records)
labels_by_id = index_by_ticket(label_records)
production_ids = [record["ticket_id"] for record in production_records]

assert set(predictions_by_id) == set(labels_by_id) == set(production_ids)

actual = [labels_by_id[ticket_id]["queue"] for ticket_id in production_ids]
predicted = [
    predictions_by_id[ticket_id]["predicted_queue"]
    for ticket_id in production_ids
]
queue_labels = monitoring_job_config["queue_labels"]

verified_metrics = checks_module.classification_metrics(
    actual, predicted, queue_labels
)

saved_quality = json.loads(
    read_saved_s3_bytes(project_config["monitoring"]["production_quality"])
)
assert saved_quality["batch_job_name"] == monitoring_job_config["batch_job_name"]

for metric, value in verified_metrics.items():
    assert math.isclose(
        value, saved_quality["metrics"][metric],
        rel_tol=1e-9, abs_tol=1e-12,
    ), f"Metric mismatch: {metric}"
    print(f"PASS: {metric} matches the saved result ({value:.4f}).")

# Independently check the formulas with a small, known example.
# Class C has no examples but must still count in macro F1.
example_metrics = checks_module.classification_metrics(
    ["A", "A", "B", "B"],
    ["A", "A", "A", "A"],
    ["A", "B", "C"],
)
for metric, expected in {
    "accuracy": 0.5,
    "macro_f1": 2 / 9,
    "weighted_f1": 1 / 3,
}.items():
    assert math.isclose(example_metrics[metric], expected, abs_tol=1e-12)

print("PASS: metric formulas handle an absent class correctly.")

# Make every prediction wrong in a separate test.
next_queue = {
    queue: queue_labels[(index + 1) % len(queue_labels)]
    for index, queue in enumerate(queue_labels)
}
bad_predictions = [next_queue[queue] for queue in actual]

bad_metrics = checks_module.classification_metrics(
    actual, bad_predictions, queue_labels
)
bad_model_checks = checks_module.model_quality_checks(
    bad_metrics, reference_metrics, allowed_drop
)

assert all(value == 0.0 for value in bad_metrics.values())
assert all(check["status"] == "ALERT" for check in bad_model_checks)

print("PASS: deliberately wrong predictions triggered all 3 quality alerts.")

verification_report = {
    "verified_at": datetime.now(timezone.utc).isoformat(),
    "checks_script_sha256": hashlib.sha256(checks_path.read_bytes()).hexdigest(),
    "production_rows": len(production_ids),
    "production_metrics": verified_metrics,
    "tests_passed": [
        "Data checks match saved notebook results",
        "Empty text and unfamiliar language are detected",
        "Accuracy and F1 match saved production results",
        "Known metric example with an absent class",
        "Incorrect predictions trigger all model quality alerts",
    ],
}

(monitoring_report_dir / "reusable_checks_verification.json").write_text(
    json.dumps(verification_report, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Verification report saved. No synthetic results published to AWS.")

PASS: accuracy matches the saved result (0.4591).
PASS: macro_f1 matches the saved result (0.4380).
PASS: weighted_f1 matches the saved result (0.4605).
PASS: metric formulas handle an absent class correctly.
PASS: deliberately wrong predictions triggered all 3 quality alerts.
Verification report saved. No synthetic results published to AWS.


In [4]:
%%writefile src/ticket_monitoring/run_monitoring.py
"""Run ticket monitoring and save reports in a SageMaker Processing job."""

import argparse
import csv
import hashlib
import io
import json
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import boto3

from checks import (
    classification_metrics,
    data_quality_checks,
    model_quality_checks,
)


def require(condition, message):
    if not condition:
        raise ValueError(message)


def read_saved_bytes(client, source):
    location = urlparse(source["uri"])
    version = source["version_id"]
    require(version and version != "null", "Missing S3 version ID.")

    response = client.get_object(
        Bucket=location.netloc,
        Key=location.path.lstrip("/"),
        VersionId=version,
    )
    require(response["VersionId"] == version, "S3 version mismatch.")

    with response["Body"] as body:
        content = body.read()

    if source.get("sha256"):
        require(
            hashlib.sha256(content).hexdigest() == source["sha256"],
            "File fingerprint mismatch.",
        )
    return content


def read_csv_records(content):
    return list(csv.DictReader(io.StringIO(content.decode("utf-8-sig"))))


def index_records(records):
    result = {}
    for record in records:
        ticket_id = record.get("ticket_id")
        require(isinstance(ticket_id, str) and ticket_id.strip(),
                "Missing or invalid ticket ID.")
        require(ticket_id not in result, "Duplicate ticket ID.")
        result[ticket_id] = record
    return result


def summarize_job(details, kind):
    prefix = "Training" if kind == "training" else "Transform"
    status = details[f"{prefix}JobStatus"]
    start = details.get(f"{prefix}StartTime")
    end = details.get(f"{prefix}EndTime")

    state = (
        "OK" if status == "Completed"
        else "ALERT" if status in {"Failed", "Stopped", "Stopping"}
        else "PENDING"
    )

    return {
        "job_type": kind,
        "job_name": details[f"{prefix}JobName"],
        "aws_status": status,
        "check_status": state,
        "duration_seconds": (
            (end - start).total_seconds() if start and end else None
        ),
        "start_time": start.isoformat() if start else None,
        "end_time": end.isoformat() if end else None,
        "failure_reason": details.get("FailureReason"),
    }


def alert_count(checks):
    return sum(item["status"] == "ALERT" for item in checks)


def save_json(folder, filename, value):
    (folder / filename).write_text(
        json.dumps(value, indent=2, allow_nan=False),
        encoding="utf-8",
    )


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--config",
        default=str(Path(__file__).with_name("monitoring_config.json")),
    )
    parser.add_argument("--output-dir", default="/opt/ml/processing/output")
    parser.add_argument("--publish-metrics", action="store_true")
    args = parser.parse_args()

    config = json.loads(Path(args.config).read_text(encoding="utf-8"))
    output = Path(args.output_dir)
    output.mkdir(parents=True, exist_ok=True)

    session = boto3.Session(region_name=config["aws_region"])
    s3 = session.client("s3")
    sagemaker = session.client("sagemaker")

    baseline = json.loads(read_saved_bytes(s3, config["training_baseline"]))
    records = read_csv_records(read_saved_bytes(s3, config["production_data"]))

    require(len(records) == config["expected_production_rows"],
            "Unexpected production row count.")
    require(all("queue" not in row for row in records),
            "Production inputs contain queue labels.")
    production = index_records(records)

    predictions = []
    for source in config["prediction_artifacts"]:
        content = read_saved_bytes(s3, source).decode("utf-8")
        predictions.extend(
            json.loads(line) for line in content.splitlines() if line.strip()
        )
    predictions = index_records(predictions)

    labels = index_records(
        read_csv_records(read_saved_bytes(s3, config["production_labels"]))
    )

    require(set(production) == set(predictions) == set(labels),
            "Input, prediction, and label ticket IDs do not match.")

    ticket_ids = list(production)
    actual = [labels[ticket_id]["queue"] for ticket_id in ticket_ids]
    predicted = [
        predictions[ticket_id]["predicted_queue"] for ticket_id in ticket_ids
    ]

    data_checks = data_quality_checks(
        records, baseline, config["data_thresholds"]
    )
    metrics = classification_metrics(actual, predicted, config["queue_labels"])
    model_checks = model_quality_checks(
        metrics,
        config["reference_metrics"],
        config["model_allowed_absolute_drop"],
    )

    jobs = [
        summarize_job(
            sagemaker.describe_training_job(
                TrainingJobName=config["training_job_name"]
            ),
            "training",
        ),
        summarize_job(
            sagemaker.describe_transform_job(
                TransformJobName=config["batch_job_name"]
            ),
            "batch_prediction",
        ),
    ]

    now = datetime.now(timezone.utc)
    infrastructure_alerts = sum(
        job["check_status"] == "ALERT" for job in jobs
    )

    save_json(output, "production_data_quality.json", {
        "reference_source": baseline["source"],
        "production_source": config["production_data"],
        "production_rows": len(records),
        "thresholds": config["data_thresholds"],
        "checks": data_checks,
        "alert_count": alert_count(data_checks),
    })
    save_json(output, "production_quality.json", {
        "batch_job_name": config["batch_job_name"],
        "ground_truth_source": config["production_labels"],
        "metrics": {
            "evaluation_split": "simulated_production",
            "tickets": len(records),
            **metrics,
        },
    })
    save_json(output, "model_quality_checks.json", {
        "batch_job_name": config["batch_job_name"],
        "reference_metrics": config["reference_metrics"],
        "allowed_absolute_drop": config["model_allowed_absolute_drop"],
        "checks": model_checks,
        "alert_count": alert_count(model_checks),
    })
    save_json(output, "infrastructure_status.json", {
        "checked_at": now.isoformat(),
        "jobs": jobs,
        "alert_count": infrastructure_alerts,
    })
    save_json(output, "monitoring_config.json", config)

    metric_data = []

    def add_metric(name, value, unit="None"):
        metric_data.append({
            "MetricName": name,
            "Dimensions": config["cloudwatch"]["dimensions"],
            "Timestamp": now,
            "Value": float(value),
            "Unit": unit,
        })

    for metric, name in [
        ("accuracy", "ProductionAccuracy"),
        ("macro_f1", "ProductionMacroF1"),
        ("weighted_f1", "ProductionWeightedF1"),
    ]:
        add_metric(name, metrics[metric])

    add_metric("DataAlertCount", alert_count(data_checks), "Count")
    add_metric("ModelQualityAlertCount", alert_count(model_checks), "Count")
    add_metric("InfrastructureAlertCount", infrastructure_alerts, "Count")

    metric_names = {
        "language_distribution": "LanguageDistributionDistance",
        "text_char_count_distribution": "TextCharDistributionDistance",
        "text_word_count_distribution": "TextWordDistributionDistance",
        "missing_subject_rate_increase": "MissingSubjectRateIncrease",
        "missing_body_rate_increase": "MissingBodyRateIncrease",
        "empty_text_rate": "EmptyTextRate",
        "unknown_language_rate": "UnknownLanguageRate",
        "unseen_language_rate": "UnseenLanguageRate",
    }
    for check in data_checks:
        add_metric(metric_names[check["check"]], check["value"])

    for job in jobs:
        prefix = "Training" if job["job_type"] == "training" else "Batch"
        if job["duration_seconds"] is not None:
            add_metric(f"{prefix}DurationSeconds",
                       job["duration_seconds"], "Seconds")
        add_metric(f"{prefix}JobAlert",
                   int(job["check_status"] == "ALERT"), "Count")

    if args.publish_metrics:
        response = session.client("cloudwatch").put_metric_data(
            Namespace=config["cloudwatch"]["namespace"],
            MetricData=metric_data,
        )
        require(response["ResponseMetadata"]["HTTPStatusCode"] == 200,
                "CloudWatch metric publication failed.")

    summary = {
        "checked_at": now.isoformat(),
        "batch_job_name": config["batch_job_name"],
        "production_rows": len(records),
        "metrics": metrics,
        "data_alert_count": alert_count(data_checks),
        "model_alert_count": alert_count(model_checks),
        "infrastructure_alert_count": infrastructure_alerts,
        "cloudwatch_metrics_published": args.publish_metrics,
        "metric_count": len(metric_data) if args.publish_metrics else 0,
    }
    save_json(output, "monitoring_summary.json", summary)
    print(json.dumps(summary, indent=2, allow_nan=False))


if __name__ == "__main__":
    main()

Writing src/ticket_monitoring/run_monitoring.py


In [5]:
import subprocess
import sys

# Check both Python files for syntax errors.
for filename in ["checks.py", "run_monitoring.py"]:
    script_path = MONITORING_SOURCE_DIR / filename
    compile(
        script_path.read_text(encoding="utf-8"),
        str(script_path),
        "exec",
    )

print("Both scripts passed the syntax check.")

local_output_dir = PROJECT_ROOT / "data" / "monitoring_preflight"

result = subprocess.run(
    [
        sys.executable,
        str(MONITORING_SOURCE_DIR / "run_monitoring.py"),
        "--config",
        str(MONITORING_SOURCE_DIR / "monitoring_config.json"),
        "--output-dir",
        str(local_output_dir),
    ],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=180,
)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    result.check_returncode()

local_summary = json.loads(
    (local_output_dir / "monitoring_summary.json").read_text(
        encoding="utf-8"
    )
)

assert local_summary["production_rows"] == len(production_records)
assert local_summary["cloudwatch_metrics_published"] is False

for metric, expected in verified_metrics.items():
    assert math.isclose(
        local_summary["metrics"][metric],
        expected,
        rel_tol=1e-9,
        abs_tol=1e-12,
    ), f"Result mismatch: {metric}"

print(json.dumps(local_summary, indent=2))
print("\nPASS: the complete script reproduces the verified model metrics.")
print("No new AWS jobs started and no CloudWatch metrics published.")

Both scripts passed the syntax check.
{
  "checked_at": "2026-10-04T14:37:51.544498+00:00",
  "batch_job_name": "aai540-group3-batch-20261004-045734",
  "production_rows": 17712,
  "metrics": {
    "accuracy": 0.4591237579042457,
    "macro_f1": 0.4379727744255188,
    "weighted_f1": 0.4605194849461991
  },
  "data_alert_count": 0,
  "model_alert_count": 0,
  "infrastructure_alert_count": 0,
  "cloudwatch_metrics_published": false,
  "metric_count": 0
}

PASS: the complete script reproduces the verified model metrics.
No new AWS jobs started and no CloudWatch metrics published.


In [6]:
from botocore.validate import validate_parameters

sagemaker_client = aws_session.client("sagemaker")

# Prevent accidentally replacing an existing job configuration.
existing_job = project_config["monitoring"].get("processing_job", {})
assert not existing_job.get("job_name"), (
    "A monitoring job is already configured. Share its status before continuing."
)

source_names = [
    "checks.py",
    "run_monitoring.py",
    "monitoring_config.json",
]

source_contents = {
    name: (MONITORING_SOURCE_DIR / name).read_bytes()
    for name in source_names
}
source_hashes = {
    name: hashlib.sha256(content).hexdigest()
    for name, content in source_contents.items()
}

package_id = hashlib.sha256(
    json.dumps(source_hashes, sort_keys=True).encode("utf-8")
).hexdigest()[:16]

code_prefix = f"monitoring-code/{package_id}/"
source_artifacts = {}

for name, content in source_contents.items():
    key = code_prefix + name
    uploaded = s3_client.put_object(
        Bucket=PROJECT_BUCKET,
        Key=key,
        Body=content,
    )
    version_id = uploaded.get("VersionId")
    assert version_id and version_id != "null"

    source_artifacts[name] = {
        "uri": f"s3://{PROJECT_BUCKET}/{key}",
        "version_id": version_id,
        "sha256": source_hashes[name],
    }

# Reuse the execution role and container from successful managed training.
training_details = sagemaker_client.describe_training_job(
    TrainingJobName=monitoring_job_config["training_job_name"]
)
processing_role = training_details["RoleArn"]
processing_image = training_details["AlgorithmSpecification"]["TrainingImage"]

processing_job_name = (
    "aai540-group3-monitor-"
    + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
)
processing_output_uri = (
    f"s3://{PROJECT_BUCKET}/monitoring-jobs/{processing_job_name}/reports/"
)

processing_request = {
    "ProcessingJobName": processing_job_name,
    "RoleArn": processing_role,
    "AppSpecification": {
        "ImageUri": processing_image,
        "ContainerEntrypoint": [
            "python3",
            "/opt/ml/processing/code/run_monitoring.py",
        ],
        "ContainerArguments": [
            "--config", "/opt/ml/processing/code/monitoring_config.json",
            "--output-dir", "/opt/ml/processing/output",
            "--publish-metrics",
        ],
    },
    "ProcessingInputs": [{
        "InputName": "monitoring-code",
        "S3Input": {
            "S3Uri": f"s3://{PROJECT_BUCKET}/{code_prefix}",
            "LocalPath": "/opt/ml/processing/code",
            "S3DataType": "S3Prefix",
            "S3InputMode": "File",
            "S3DataDistributionType": "FullyReplicated",
        },
    }],
    "ProcessingOutputConfig": {
        "Outputs": [{
            "OutputName": "monitoring-reports",
            "S3Output": {
                "S3Uri": processing_output_uri,
                "LocalPath": "/opt/ml/processing/output",
                "S3UploadMode": "EndOfJob",
            },
        }],
    },
    "ProcessingResources": {
        "ClusterConfig": {
            "InstanceCount": 1,
            "InstanceType": "ml.m5.xlarge",
            "VolumeSizeInGB": 20,
        },
    },
    "StoppingCondition": {"MaxRuntimeInSeconds": 900},
}

# Validate the request structure without submitting it.
validate_parameters(
    processing_request,
    sagemaker_client.meta.service_model.operation_model(
        "CreateProcessingJob"
    ).input_shape,
)

request_path = monitoring_report_dir / "processing_job_request.json"
request_path.write_text(
    json.dumps(processing_request, indent=2),
    encoding="utf-8",
)

project_config["monitoring"]["processing_job"] = {
    "job_name": processing_job_name,
    "status": "Prepared",
    "image_uri": processing_image,
    "role_arn": processing_role,
    "output_uri": processing_output_uri,
    "source_artifacts": source_artifacts,
    "request_file": str(request_path.relative_to(PROJECT_ROOT)),
}

config_path.write_text(
    json.dumps(project_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Monitoring source files uploaded:", len(source_artifacts))
print("Job name:", processing_job_name)
print("Compute: one ml.m5.xlarge instance")
print("Maximum runtime: 15 minutes")
print("Job configuration validated and saved. Job not started.")

Monitoring source files uploaded: 3
Job name: aai540-group3-monitor-20261004-143912
Compute: one ml.m5.xlarge instance
Maximum runtime: 15 minutes
Job configuration validated and saved. Job not started.


In [7]:
# Reload saved state to prevent duplicate submissions.
project_config = json.loads(config_path.read_text(encoding="utf-8"))
processing_info = project_config["monitoring"]["processing_job"]
processing_job_name = processing_info["job_name"]

if processing_info["status"] == "Prepared":
    # Confirm the uploaded source files have not changed.
    for source in processing_info["source_artifacts"].values():
        location = urlparse(source["uri"])
        current = s3_client.head_object(
            Bucket=location.netloc,
            Key=location.path.lstrip("/"),
        )
        assert current["VersionId"] == source["version_id"], (
            "An uploaded source file changed."
        )

    processing_request = json.loads(
        (PROJECT_ROOT / processing_info["request_file"])
        .read_text(encoding="utf-8")
    )
    assert processing_request["ProcessingJobName"] == processing_job_name

    # Save the submission state before contacting AWS.
    processing_info["status"] = "Submitting"
    config_path.write_text(
        json.dumps(project_config, indent=2),
        encoding="utf-8",
    )

    submitted = sagemaker_client.create_processing_job(**processing_request)
    processing_info["job_arn"] = submitted["ProcessingJobArn"]
else:
    print("Submission already recorded; checking the existing job.")

processing_details = sagemaker_client.describe_processing_job(
    ProcessingJobName=processing_job_name
)

processing_info["status"] = processing_details["ProcessingJobStatus"]
config_path.write_text(
    json.dumps(project_config, indent=2),
    encoding="utf-8",
)

print("Monitoring job:", processing_job_name)
print("Status:", processing_info["status"])
print("Report destination:", processing_info["output_uri"])

Monitoring job: aai540-group3-monitor-20261004-143912
Status: InProgress
Report destination: s3://aai540-group3-tickets-390568313988-us-east-1/monitoring-jobs/aai540-group3-monitor-20261004-143912/reports/


In [10]:
processing_details = sagemaker_client.describe_processing_job(
    ProcessingJobName=processing_job_name
)

status = processing_details["ProcessingJobStatus"]
project_config["monitoring"]["processing_job"]["status"] = status

config_path.write_text(
    json.dumps(project_config, indent=2),
    encoding="utf-8",
)

print("Monitoring job:", processing_job_name)
print("Current status:", status)

if status == "Completed":
    print("Reports saved:", processing_info["output_uri"])
elif status in {"Failed", "Stopped"}:
    print("Reason:", processing_details.get("FailureReason", "Not provided"))
else:
    print("Still running. Run this status cell again in about one minute.")

Monitoring job: aai540-group3-monitor-20261004-143912
Current status: Completed
Reports saved: s3://aai540-group3-tickets-390568313988-us-east-1/monitoring-jobs/aai540-group3-monitor-20261004-143912/reports/


In [11]:
report_names = [
    "production_data_quality.json",
    "production_quality.json",
    "model_quality_checks.json",
    "infrastructure_status.json",
    "monitoring_config.json",
    "monitoring_summary.json",
]

output_location = urlparse(processing_info["output_uri"])
output_prefix = output_location.path.lstrip("/").rstrip("/") + "/"

managed_reports = {}
report_contents = {}
report_artifacts = {}

for filename in report_names:
    key = output_prefix + filename
    response = s3_client.get_object(
        Bucket=output_location.netloc,
        Key=key,
    )

    version_id = response.get("VersionId")
    assert version_id and version_id != "null"

    with response["Body"] as body:
        content = body.read()

    managed_reports[filename] = json.loads(content)
    report_contents[filename] = content
    report_artifacts[Path(filename).stem] = {
        "uri": f"s3://{output_location.netloc}/{key}",
        "version_id": version_id,
        "sha256": hashlib.sha256(content).hexdigest(),
    }

managed_summary = managed_reports["monitoring_summary.json"]

assert managed_summary["batch_job_name"] == (
    monitoring_job_config["batch_job_name"]
)
assert managed_summary["production_rows"] == len(production_records)
assert managed_summary["cloudwatch_metrics_published"] is True
assert managed_summary["metric_count"] == 18
assert managed_reports["monitoring_config.json"] == monitoring_job_config

# Verify model scores match the locally verified calculations.
for metric, expected in verified_metrics.items():
    assert math.isclose(
        managed_summary["metrics"][metric],
        expected,
        rel_tol=1e-9,
        abs_tol=1e-12,
    ), f"Managed metric mismatch: {metric}"

# Verify all data checks match, allowing tiny floating-point differences.
expected_data_checks = {
    item["check"]: item for item in verified_data_checks
}
managed_data_checks = managed_reports["production_data_quality.json"]["checks"]

assert {item["check"] for item in managed_data_checks} == set(
    expected_data_checks
)

for check in managed_data_checks:
    expected = expected_data_checks[check["check"]]
    assert check["status"] == expected["status"]
    assert check["threshold"] == expected["threshold"]
    assert math.isclose(
        check["value"], expected["value"],
        rel_tol=1e-9, abs_tol=1e-12,
    )

# Save the verified reports locally for GitHub and the final report.
managed_report_dir = PROJECT_ROOT / "reports" / "managed_monitoring"
managed_report_dir.mkdir(parents=True, exist_ok=True)

for filename, content in report_contents.items():
    (managed_report_dir / filename).write_bytes(content)

processing_info = project_config["monitoring"]["processing_job"]
processing_info["report_artifacts"] = report_artifacts
processing_info["summary"] = managed_summary

project_config["monitoring"]["cloudwatch"]["last_published_at"] = (
    managed_summary["checked_at"]
)

config_path.write_text(
    json.dumps(project_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

print(json.dumps(managed_summary, indent=2))
print("\nPASS: managed model metrics and data checks match local results.")
print("Saved reports:", managed_report_dir)

{
  "checked_at": "2026-10-04T14:41:32.148403+00:00",
  "batch_job_name": "aai540-group3-batch-20261004-045734",
  "production_rows": 17712,
  "metrics": {
    "accuracy": 0.4591237579042457,
    "macro_f1": 0.4379727744255188,
    "weighted_f1": 0.4605194849461991
  },
  "data_alert_count": 0,
  "model_alert_count": 0,
  "infrastructure_alert_count": 0,
  "cloudwatch_metrics_published": true,
  "metric_count": 18
}

PASS: managed model metrics and data checks match local results.
Saved reports: /home/sagemaker-user/AAI-540-Project/multilingual-it-ticket-routing-mlops/reports/managed_monitoring


In [12]:
from datetime import timedelta

cloudwatch_client = aws_session.client("cloudwatch")
cloudwatch_config = project_config["monitoring"]["cloudwatch"]
dashboard_name = cloudwatch_config["dashboard_name"]

dashboard_path = monitoring_report_dir / "cloudwatch_dashboard.json"
dashboard_body = json.loads(dashboard_path.read_text(encoding="utf-8"))

published_at = datetime.fromisoformat(managed_summary["checked_at"])
start_text = (published_at - timedelta(minutes=5)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)
end_text = (published_at + timedelta(minutes=5)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)

dashboard_body["start"] = start_text
dashboard_body["end"] = end_text

for widget in dashboard_body["widgets"]:
    if widget["type"] == "metric":
        widget["properties"].update({
            "start": start_text,
            "end": end_text,
            "setPeriodToTimeRange": True,
            "sparkline": False,
        })

dashboard_body["widgets"][0]["properties"]["markdown"] = (
    "# Multilingual ticket routing — monitoring\n"
    f"**Monitoring job:** `{processing_info['job_name']}` — Completed\n\n"
    f"**Prediction batch:** `{managed_summary['batch_job_name']}`\n\n"
    f"**Snapshot published:** `{managed_summary['checked_at']}`\n\n"
    "Reports and metrics were generated by a SageMaker Processing job. "
    "This dashboard shows that run's snapshot. Recurring execution is "
    "not yet configured. Quality scores and data rates use a 0–1 scale. "
    "Zero alerts means the configured checks passed; it does not establish "
    "readiness for automatic routing."
)

dashboard_json = json.dumps(dashboard_body, indent=2)

response = cloudwatch_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=dashboard_json,
)

messages = response.get("DashboardValidationMessages", [])
if messages:
    raise RuntimeError(json.dumps(messages, indent=2))

dashboard_path.write_text(dashboard_json, encoding="utf-8")

cloudwatch_config.update({
    "dashboard_window_start": start_text,
    "dashboard_window_end": end_text,
    "dashboard_processing_job": processing_info["job_name"],
})

config_path.write_text(
    json.dumps(project_config, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Dashboard updated to the managed monitoring run.")
print("Snapshot time:", managed_summary["checked_at"])
print(cloudwatch_config["dashboard_url"])

Dashboard updated to the managed monitoring run.
Snapshot time: 2026-10-04T14:41:32.148403+00:00
https://console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=AAI540-Group3-Ticket-Monitoring
